# Assignment 5
## Data Preprocessing

We will be using a dataframe created from *Income Dirty Data.csv*. Download the file from D2L. 

1. Import the following modules
    - `pandas`
    - `numpy`
    - `preprocessing` from `sklearn` (for bonus question)
    - `KNNImputer` from `sklearn.impute` (for bonus question)
2. Create your dataframe from the file using `pandas`

In [35]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder



3. Calculate and display the following information
    - Total number of NaN values for **each column**
    - Percentage of NaN values in the dataset 
    - Number of rows *without* any NaN values

In [36]:
df= pd.read_csv("Income Dirty Data.csv")
nan_per_column = df.isna().sum()
print(nan_per_column)

total_nan = df.isna().sum().sum()
total_cells = df.size
percent_nan = (total_nan / total_cells) * 100
print(f"{percent_nan:.2f}%")

rows_without_nan = df.dropna().shape[0]
print(rows_without_nan)

ID              0
sex            88
age             0
income        109
tax_15_pct     93
dtype: int64
5.80%
733


Besides missing values (NaN), the dataset contains errors. We have the following rules to check:

* All employees are adults (18+ years old)
* All employees pay 15% of their income for the tax
* All employees make money; no income should be <= 0 
---
4. Calculate and display the percentage of the data that does **NOT** violate any **one** of the rules.

In [37]:
# true = follows the rule, false = violates it
rule_age = df['age'] >= 18
rule_tax = df['tax_15_pct'] == (0.15 * df['income'])
rule_income = df['income'] > 0

# A row is valid only if it passes ALL three rules
valid_rows = rule_age & rule_tax & rule_income

# Count and calculate percentage
num_valid = valid_rows.sum()
percent_valid = (num_valid / len(df)) * 100

print(f"{percent_valid:.2f}%")

47.00%


Now that we have determined the number of erroneous datapoints in our set, let's work on correcting it as best we can.

5. Replace non *Female*/*Male* values in the **Sex** column with either *Female* or *Male* (e.g., Women --> Female)

In [38]:
sex_map = {
    'Women': 'Female',
    'Woman': 'Female',
    'Man': 'Male',
    'Men': 'Male'
}

df['sex'] = df['sex'].replace(sex_map)


print(df['sex'].unique())


<StringArray>
['Female', nan, 'Male']
Length: 3, dtype: str


6. Replace non-positive **Age** values with NaN (`numpy.NaN`)
7. Replace non-positive **Income** values with NaN (`numpy.NaN`)
8. Replace non-positive **Tax (15%)** values with NaN (`numpy.NaN`)

In [39]:

df.loc[df['age'] <= 0, 'age'] = np.nan


df.loc[df['income'] <= 0, 'income'] = np.nan


df.loc[df['tax_15_pct'] <= 0, 'tax_15_pct'] = np.nan

The following question is a bonus (+10) question, but I'd encourage you to give it a try!

9. Use machine learning (`KNNImputer`) to impute all missing values (replaces NaN values with the most predicted values)
    - Will need to use a scaler and convert the values in the **Sex** column to a numeric value for algorithm to work properly
    - Show some of the data prior to imputing, and after imputing

In [40]:
print("BEFORE imputing:")
print(df.head(10))
print(df.isna().sum())


le = LabelEncoder()

sex_not_null = df['sex'].notna()
df.loc[sex_not_null, 'sex_encoded'] = le.fit_transform(df.loc[sex_not_null, 'sex'])

# Select the numeric columns to impute
# ------------------------------------------------------------------
cols_to_impute = ['age', 'income', 'tax_15_pct', 'sex_encoded']
data_to_impute = df[cols_to_impute]


scaler = StandardScaler()
scaled_data = scaler.fit_transform(data_to_impute)


imputer = KNNImputer(n_neighbors=5)
imputed_array = imputer.fit_transform(scaled_data)


unscaled_data = scaler.inverse_transform(imputed_array)
df_imputed = pd.DataFrame(unscaled_data, columns=cols_to_impute, index=df.index)


df['age'] = df_imputed['age']
df['income'] = df_imputed['income']
df['tax_15_pct'] = df_imputed['tax_15_pct']
df['sex_encoded'] = df_imputed['sex_encoded'].round().astype(int)


df['sex'] = le.inverse_transform(df['sex_encoded'])
df = df.drop(columns=['sex_encoded'])


print("\nAFTER imputing:")
print(df.head(10))
print(df.isna().sum())

BEFORE imputing:
   ID     sex   age    income  tax_15_pct
0   1  Female  21.0  147168.0    22075.20
1   2  Female  29.0  119595.0    17939.25
2   3  Female  56.0   87770.0    13165.50
3   4     NaN  21.0   54259.0     8138.85
4   5    Male  28.0       NaN   160230.00
5   6  Female   NaN  128326.0    19248.90
6   7  Female   NaN       NaN         NaN
7   8  Female  24.0       NaN    11820.60
8   9  Female  38.0  149473.0    22420.95
9  10    Male  48.0  113663.0  1136630.00
ID              0
sex            88
age            98
income        153
tax_15_pct    103
dtype: int64

AFTER imputing:
   ID     sex   age    income  tax_15_pct
0   1  Female  21.0  147168.0    22075.20
1   2  Female  29.0  119595.0    17939.25
2   3  Female  56.0   87770.0    13165.50
3   4    Male  21.0   54259.0     8138.85
4   5    Male  28.0   87077.0   160230.00
5   6  Female  36.2  128326.0    19248.90
6   7  Female  50.6  114249.4    15496.35
7   8  Female  24.0  110944.4    11820.60
8   9  Female  38.0  14

10. In the empty `Markdown` cell below, explain why it is important to clean a dataset before calculating analytics about the data


Dirty data (missing values, impossible entries, inconsistent formatting) skews calculations like means and percentages, leading to inaccurate or misleading analytics. Cleaning the dataset first ensures that any inferences or decisions drawn from the data are based on accurate, trustworthy information.

### Submission to D2L Dropbox
- Submit this `Jupyter` file to D2L, renamed as **Last_First_Assignment5.ipynb** 
    - Replace '**Last**' and '**First**' with your first and last name

- Include the link to your GitHub repository (the URL of your repo page, for
   example `https://github.com/yourname/csci-4047-work`).
### How to add my code to GitHub?

1. Stage your file (this tells Git which changes to include):

      `git add Last_First_Assignment5.ipynb`

   To stage everything (you might not want to stage everything though) in the folder instead, use `git add .`

3. Commit your changes (this saves a snapshot with a message):

       git commit -m "Add Assignment 5"

   The text in quotes is your commit message. Make it describe what you
   did.

4. Push your commit up to GitHub:

       git push -u origin main

   The `-u origin main` part is only needed the first push. After that,
   `git push` alone is enough.

**What each command does, briefly**

- `git add` picks which files to include in the next save.
- `git commit` saves a snapshot of those files on your computer, with a
  message describing the change.
- `git push` uploads your saved commits to GitHub so they appear online.